<a href="https://colab.research.google.com/github/magonz3003-eng/Motor_Predictive_Maintenance_Data/blob/main/RX3i_RAG_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install -U langchain-classic langchain langchain-community langchain-openai

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load your manual
# IMPORTANT: Click the folder icon on the left and upload your PDF first!
file_path = "/content/GFK-2314T_RX3i_SystemManual.pdf"
loader = PyPDFLoader(file_path)
data = loader.load()

# 2. Split it into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
docs = text_splitter.split_documents(data)

print(f"✅ Success! Manual loaded and broken into {len(docs)} chunks.")

✅ Success! Manual loaded and broken into 1985 chunks.


In [3]:
import getpass
import os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API Key here: ")

Paste your OpenAI API Key here: ··········


In [25]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. Setup the Searcher
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

# 2. Setup the Brain
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# 3. The Instructions
system_prompt = (
    "You are an expert industrial maintenance assistant. "
    "Use the technical context below to answer the user's question. "
    "If you don't know, say you don't know.\n\n{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# 4. Connect the dots using the CLASSIC paths
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("🚀 DONE. The classic engine is finally running!")

🚀 DONE. The classic engine is finally running!


In [26]:
# Ask a specific technical question
query = "What are the LED status indicators for a CPU fault on the RX3i?"
response = rag_chain.invoke({"input": query})

print(f"QUESTION: {query}")
print("-" * 30)
print(f"MANUAL ANSWER: {response['answer']}")

QUESTION: What are the LED status indicators for a CPU fault on the RX3i?
------------------------------
MANUAL ANSWER: The PACSystems* RX3i System Manual does not specifically mention LED status indicators for a CPU fault. However, typically in industrial systems, a CPU fault may be indicated by a red LED on the CPU module itself or through a system-level fault indicator. It is recommended to refer to the specific CPU module's manual or contact the manufacturer for detailed information on CPU fault LED status indicators in the RX3i system.


In [29]:
query = "What are the configuration steps for setting up serial interface module?"
response = rag_chain.invoke({"input": query})

print(f"QUESTION: {query}")
print("-" * 30)
print(f"MANUAL ANSWER: {response['answer']}")


QUESTION: What are the configuration steps for setting up serial interface module?
------------------------------
MANUAL ANSWER: To set up a serial interface module, follow these general configuration steps:

1. **Physical Connection**: Ensure the module is correctly installed in the system and connected to the device you want to communicate with using the appropriate cables.

2. **Configuration Software**: Use the appropriate configuration software provided by the manufacturer to access the module's settings. This can be done through a serial port on the host CPU or a web-based tool if the CPU lacks a serial port.

3. **Firmware Upgrade**: If necessary, upgrade the module's firmware using the kit provided. Follow the instructions included in the firmware upgrade kit for this process.

4. **Refer to Manuals**: For detailed information on configuring the module, refer to the following publications:
   - PACSystems RX3i and RSTi-EP TCP/IP Ethernet Communications User Manual, GFK-2224
   